In [ ]:
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
load_dotenv()

class Outputs:
    def __init__(self, response):
        self.response =  response
    def get_text(self):
        content = self.response
        if isinstance(content, list):
            return content[0].get("text", "")
        return str(content)

class DataOrchestrator:
    def __init__(self):
        self.model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)
        self.df = None

    def process_csv(self, file_path):
        """The 'Blind' function that the AI doesn't see or touch."""
        try:
            self.df = pd.read_csv(file_path)
            return f"SUCCESS: Loaded {file_path}. Rows: {len(self.df)}, Columns: {list(self.df.columns)}"
        except Exception as e:
            return f"FAILURE: Could not load file. Error: {str(e)}"

    def run(self, user_input, file_path):
        prompt = (
            f"The user says: '{user_input}'. "
            f"They provided a file path: '{file_path}'. "
            "Task: Is this file extension .csv? Answer only 'YES' or 'NO'."
        )
        intent_check = Outputs(self.model.invoke(prompt))
        intent_check = intent_check.get_text()
        intent_check = intent_check.strip().upper()
        if "YES" in intent_check:
            # STEP 2: Python runs the code, NOT the AI
            print(f"--- Python is now processing {file_path} ---")
            result_of_function = self.process_csv(file_path)

            # STEP 3: AI interprets the output for the user
            final_prompt = (
                f"The user asked: '{user_input}'. "
                f"The Python system ran a process and returned this result: '{result_of_function}'. "
                "Please give a polite, professional and concise summary of this result to the user."
            )
            final_response = self.model.invoke(final_prompt)
            return final_response.content
        else:
            return "The file you've given me isn't a .csv. Check if you have uploaded the correct file."

# --- EXECUTION ---
orchestrator = DataOrchestrator()
user_msg = "Hey, can you please convert this file into a dataframe for me?"
file_to_check = "spesa.csv"
msg_to_validator = "The user gave us a .csv that I have already loaded in a dataframe, perform a validation check."

response_to_user = orchestrator.run(user_msg, file_to_check)
response = Outputs(response_to_user)
print(response.get_text())

if orchestrator.df is not None:
    print("\nMain Program: The DataFrame is ready for further analysis!")

In [ ]:
class SchemaValidator:
    def __init__(self):
        self.model=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)
    
    def run_validation_check(self, msg_to_validator, df):
        df = orchestrator.df
        internal_schema_info = {
            col: str(dtype) 
            for col, dtype in zip(df.columns, df.dtypes)
        }
        
        prompt = (
            f"Context: {msg_to_validator}\n"
            f"Current Schema (Column: Type): {internal_schema_info}\n\n"
            "Task: Identify columns where the data type does not match logical expectations "
            "(e.g., a 'Price' column being a string, or 'ID' being a float).\n\n"
            "Output Rules:\n"
            "1. If mismatches are found: List each column using a bullet point, followed by a brief, "
            "natural explanation of why the type is suspicious. Example: '- price: Currently a string, but should be a float.'\n"
            "2. If all match: Return ONLY 'All match'."
        )
        message = self.model.invoke(prompt)
        return message.content
    
    def run_naming_check(self, msg_to_validator, df):
        df = orchestrator.df
        prompt = (
            f"Context: {msg_to_validator}\n"
            f"Current column names: {df.columns.to_list()}\n\n"
            "Task: Identify columns where the name of the column doesn't follow a known standard naming standard or the general structure of the names of the other columns"
            "(e.g., a column using special characters, wrong casing, or reserved words). \n\n"
            "Output Rules:\n"
            "1. If issues found: List each column with a bullet point and a human-friendly explanation of the naming violation.\n"
            "2. If no issues: Return ONLY 'No convention breaks'."
        )
        message = self.model.invoke(prompt)
        return message.content

validator = SchemaValidator()
first_message_back= validator.run_validation_check(msg_to_validator, orchestrator.df)
first_message_back = Outputs(first_message_back)
second_message_back = validator.run_naming_check(msg_to_validator, orchestrator.df)
second_message_back = Outputs(second_message_back)
print(first_message_back.get_text())
print("\n")
print(second_message_back.get_text())